# California ACSIncome 2018: target MLP baseline
Run cells manually, in order. No experiments have been executed.
Before starting, install dependencies in your project environment with `.\.venv\Scripts\python.exe -m pip install -r requirements-training.txt`, then select that environment as the notebook kernel.

Default partitions: 165,665 training, 10,000 validation, 10,000 target testing, and 10,000 MIA non-members. The 10,000 MIA members are a subset of training. Adult shadow models and the separate 2017 generalization evaluation are later steps.

In [12]:
from pathlib import Path
import json
import hashlib
import importlib.metadata
import joblib
import pandas as pd
from src.data import load_acs_income, split_acs_data, preprocess_partitions, save_partition_ids
from src.training import train_target_model, evaluate_model, save_target_model

# Open this notebook from the project root. Use a new RUN_DIR per experiment.
DATA_PATH = Path('data/processed/acs_income_ca_2018.csv')
RUN_DIR = Path('artifacts/target_2018_mlp_seed42')
SEED = 42
SPLIT_CONFIG = dict(validation_size=10_000, test_size=10_000,
                   mia_nonmember_size=10_000, mia_member_size=10_000, seed=SEED)
TRAIN_CONFIG = dict(hidden_dims=(128, 64), dropout=0.0, epochs=50,
                    batch_size=256, learning_rate=1e-3, weight_decay=1e-4,
                    patience=5, seed=SEED, device='cpu')

## Load and split
IDs refer to rows of this exact CSV, not unique people. Identical feature vectors may occur naturally. Keep test and MIA partitions out of model selection.

In [39]:
data = load_acs_income(DATA_PATH)
partitions = split_acs_data(data, **SPLIT_CONFIG)
pd.DataFrame({name: {'records': len(part), 'positive_rate': part['income_gt_50k'].mean()}
              for name, part in partitions.items()}).T

,records,positive_rate
validation,10000.0,0.41060
test,10000.0,0.41060
mia_nonmembers,10000.0,0.41060
train,165665.0,0.41057
mia_members,10000.0,0.41060


## Preprocess
Age and weekly hours are standardized. Other Census codes, including education, are one-hot encoded. Missing values are imputed using training statistics; unseen categories are ignored. The original partition frames remain available for later feature masking.

In [14]:
preprocessor, arrays = preprocess_partitions(partitions)
{name: values['X'].shape for name, values in arrays.items()}

{'validation': (10000, 816),
 'test': (10000, 816),
 'mia_nonmembers': (10000, 816),
 'train': (165665, 816),
 'mia_members': (10000, 816)}

## Train
This cell trains the standard MLP and restores the epoch with the lowest validation loss. It does not train a differentially private model.

In [15]:
model, history, metadata = train_target_model(
    arrays['train']['X'], arrays['train']['y'],
    arrays['validation']['X'], arrays['validation']['y'],
    **TRAIN_CONFIG,
)
pd.DataFrame(history)

Epoch 001: train loss=0.4062, validation loss=0.3803, validation accuracy=0.8299
Epoch 002: train loss=0.3804, validation loss=0.3773, validation accuracy=0.8307
Epoch 003: train loss=0.3746, validation loss=0.3766, validation accuracy=0.8304
Epoch 004: train loss=0.3706, validation loss=0.3730, validation accuracy=0.8342
Epoch 005: train loss=0.3665, validation loss=0.3736, validation accuracy=0.8333
Epoch 006: train loss=0.3620, validation loss=0.3859, validation accuracy=0.8241
Epoch 007: train loss=0.3587, validation loss=0.3775, validation accuracy=0.8303
Epoch 008: train loss=0.3549, validation loss=0.3808, validation accuracy=0.8279
Epoch 009: train loss=0.3512, validation loss=0.3770, validation accuracy=0.8322


,epoch,train_loss,validation_accuracy,validation_roc_auc,validation_log_loss
0,1,0.406219,0.8299,0.907202,0.380297
1,2,0.380415,0.8307,0.908465,0.377303
2,3,0.374619,0.8304,0.908759,0.376627
3,4,0.370585,0.8342,0.910394,0.372956
4,5,0.366494,0.8333,0.909728,0.373620
5,6,0.361954,0.8241,0.909141,0.385946
6,7,0.358734,0.8303,0.908067,0.377454
7,8,0.354860,0.8279,0.907454,0.380815
8,9,0.351232,0.8322,0.909067,0.376958


## Final predictive evaluation
Run after fixing hyperparameters using validation only. These are income-prediction metrics, not membership-attack metrics. Do not tune the model using these results.

In [16]:
metrics = {name: evaluate_model(model, arrays[name]['X'], arrays[name]['y'])
           for name in ['train', 'validation', 'test']}
pd.DataFrame(metrics).T

,accuracy,roc_auc,log_loss
train,0.83786,0.916747,0.361657
validation,0.83420,0.910394,0.372956
test,0.83350,0.907860,0.378340


## Preliminary 2017 evaluation
Run after training the 2018 model in this kernel. These cells use all 192,929 California 2017 ACSIncome records and the existing 2018 training preprocessor, without fitting or retraining.

Before transforming 2017 records, map OCCP to the 2018 classification using official Census TP78 tables E1 and F1. The 2018 data, fitted preprocessor, and target model remain unchanged. Direct mappings are exact code correspondences; one-to-many splits use the highest total conversion rate and are approximate individual assignments. Codes absent from the change tables retain their original values. Inspect the mapping audit and unseen-category report. Both years keep nominal income >$50,000 without inflation adjustment. Keep these results out of hyperparameter selection. Source: https://www.census.gov/data/tables/time-series/demo/industry-occupation/acs-tp78.html

In [21]:
import importlib
import src.data

importlib.reload(src.data)
import importlib
import src.data
importlib.reload(src.data)
from src.data import transform_external_data
from src.occupation_mapping import map_2017_occupations
data_2017 = load_acs_income('data/processed/acs_income_ca_2017.csv')
data_2017_mapped, occupation_mapping_audit = map_2017_occupations(data_2017)
display(occupation_mapping_audit.groupby('status')['records'].sum().to_frame())
display(occupation_mapping_audit)
arrays_2017, category_coverage_2017 = transform_external_data(data_2017_mapped, preprocessor)
print(f'2017 evaluation records: {len(data_2017):,}')
category_coverage_2017

,records
status,
ambiguous_modal,39319
direct,11534
unchanged_not_listed,142076


,occp_2017,occp_2018,records,status,conversion_rate,candidate_count
0,10,10,1884,unchanged_not_listed,1.0000,1
1,20,20,1148,unchanged_not_listed,1.0000,1
2,40,40,70,unchanged_not_listed,1.0000,1
3,50,51,1474,ambiguous_modal,0.5117,2
4,60,60,81,unchanged_not_listed,1.0000,1
...,...,...,...,...,...,...
472,9750,9760,46,direct,1.0000,1
473,9800,9800,45,unchanged_not_listed,1.0000,1
474,9810,9810,101,unchanged_not_listed,1.0000,1
475,9820,9825,325,ambiguous_modal,0.9806,2


2017 evaluation records: 192,929


,unseen_records,unseen_fraction
feature,,
COW,0,0.000000
SCHL,0,0.000000
MAR,0,0.000000
OCCP,65,0.000337
POBP,6,0.000031
RELP,0,0.000000
SEX,0,0.000000
RAC1P,0,0.000000


In [40]:
metrics_2017 = evaluate_model(model, arrays_2017['X'], arrays_2017['y'])
# Re-evaluate the 2018 test split using the same current model.
comparison = {
    '2018 held-out test': evaluate_model(model, arrays['test']['X'], arrays['test']['y']),
    '2017 (occupation mapped; ambiguous splits approximated)': metrics_2017,
}
pd.DataFrame(comparison).T

,accuracy,roc_auc,log_loss
2018 held-out test,0.833500,0.907860,0.378340
2017 (occupation mapped; ambiguous splits approximated),0.825096,0.906088,0.381894


## Random forest membership attack: complete-feature baseline
Run after training the original 2018 target. This uses the same attack procedure as the overfit notebook: five Adult shadows, four paper-inspired features (class-scaled logits, blind walk, entropy, cross-entropy), and a random forest membership classifier.

The attack is fitted on Adult shadow outputs only. Adult attack rows are split by record ID; thresholds are selected on Adult validation only. Feature extraction samples 2,000 members and 2,000 non-members per shadow. Target evaluation uses all 10,000 original target MIA members and 10,000 non-members, with this target's own fitted preprocessor. No overfit results are reused or compared.

Walk defaults: 20 steps, two walks, categorical change probability .05, batch size 512, seed 42. Repeated model queries can take time. The tabular walk is an adaptation, not an exact reproduction; see data/paper_attack_features.md. True income labels are assumed known. No target MIA results select settings. Run cells manually; none were executed during this edit.

In [44]:
import importlib
import src.shadow_models
import src.membership_attack
import src.paper_attack_features
importlib.reload(src.shadow_models)
importlib.reload(src.membership_attack)
importlib.reload(src.paper_attack_features)
from src.shadow_models import load_adult_data, train_shadow_models, summarize_shadow_models
from src.membership_attack import train_membership_classifier, evaluate_attack
from src.paper_attack_features import collect_shadow_paper_features, collect_target_paper_features

WALK_CONFIG = dict(max_steps=20, repeats=2, categorical_change_probability=0.05,
                   batch_size=512, seed=42)
SHADOW_RECORDS_PER_CLASS = 2000
# Train the five Adult shadows when you run this cell.
# Skip this cell if shadow_results is already loaded and these imports/settings are set.
adult_auxiliary = load_adult_data()
shadow_results = train_shadow_models(adult_auxiliary, seed=42)
summarize_shadow_models(shadow_results)

Training shadow 1/5: xgboost
Training shadow 2/5: random_forest
Training shadow 3/5: mlp
Epoch 001: train loss=0.4162, validation loss=0.3245, validation accuracy=0.8483
Epoch 002: train loss=0.3166, validation loss=0.3166, validation accuracy=0.8531
Epoch 003: train loss=0.3112, validation loss=0.3153, validation accuracy=0.8525
Epoch 004: train loss=0.3074, validation loss=0.3121, validation accuracy=0.8572
Epoch 005: train loss=0.3066, validation loss=0.3150, validation accuracy=0.8513
Epoch 006: train loss=0.3031, validation loss=0.3133, validation accuracy=0.8551
Epoch 007: train loss=0.3022, validation loss=0.3122, validation accuracy=0.8565
Epoch 008: train loss=0.3013, validation loss=0.3117, validation accuracy=0.8559
Epoch 009: train loss=0.2997, validation loss=0.3118, validation accuracy=0.8586
Epoch 010: train loss=0.2982, validation loss=0.3111, validation accuracy=0.8552
Epoch 011: train loss=0.2978, validation loss=0.3149, validation accuracy=0.8543
Epoch 012: train los

records  accuracy   roc_auc  log_loss
shadow_model        partition                                        
xgboost             train         19536  0.883241  0.941862  0.254841
                    validation     9769  0.875832  0.927968  0.277922
                    nonmembers    19537  0.871577  0.926679  0.280469
random_forest       train         19536  0.894656  0.958916  0.246325
                    validation     9769  0.865800  0.921603  0.298265
                    nonmembers    19537  0.865281  0.916223  0.305339
mlp                 train         19536  0.862664  0.920766  0.294177
                    validation     9769  0.855154  0.911535  0.311146
                    nonmembers    19537  0.857399  0.913891  0.306424
decision_tree       train         19536  0.879863  0.945487  0.238998
                    validation     9769  0.846351  0.865680  1.032059
                    nonmembers    19537  0.851871  0.871910  0.999966
logistic_regression train         19536  0.856214  0.909753  0.314372
                    validation     9769  0.849729  0.903692  0.322259
                    nonmembers    19537  0.849619  0.903744  0.321650

In [45]:
# Extract the four features by querying this notebook's fitted shadows.
adult_for_rf_queries = load_adult_data()
shadow_attack_data, rf_shadow_walk_diagnostics = collect_shadow_paper_features(
    shadow_results, adult_for_rf_queries, per_class=SHADOW_RECORDS_PER_CLASS,
    config=WALK_CONFIG, seed=42)
display(pd.DataFrame(rf_shadow_walk_diagnostics).T)
attack_result = train_membership_classifier(
    shadow_attack_data, seed=42, algorithm='random_forest', feature_set='paper')
# Adult attack diagnostics, not target membership performance.
pd.DataFrame(attack_result['metrics']).T

Attack feature extraction: 512/4,000
Attack feature extraction: 1,024/4,000
Attack feature extraction: 1,536/4,000
Attack feature extraction: 2,048/4,000
Attack feature extraction: 2,560/4,000
Attack feature extraction: 3,072/4,000
Attack feature extraction: 3,584/4,000
Attack feature extraction: 4,000/4,000
Attack feature extraction: 512/4,000
Attack feature extraction: 1,024/4,000
Attack feature extraction: 1,536/4,000
Attack feature extraction: 2,048/4,000
Attack feature extraction: 2,560/4,000
Attack feature extraction: 3,072/4,000
Attack feature extraction: 3,584/4,000
Attack feature extraction: 4,000/4,000
Attack feature extraction: 512/4,000
Attack feature extraction: 1,024/4,000
Attack feature extraction: 1,536/4,000
Attack feature extraction: 2,048/4,000
Attack feature extraction: 2,560/4,000
Attack feature extraction: 3,072/4,000
Attack feature extraction: 3,584/4,000
Attack feature extraction: 4,000/4,000
Attack feature extraction: 512/4,000
Attack feature extraction: 1,024/

,record_queries,max_record_queries,walk_censored_fraction,config
xgboost,130632,164000,0.685625,"{'max_steps': 20, 'repeats': 2, 'categorical_c..."
random_forest,138296,164000,0.769,"{'max_steps': 20, 'repeats': 2, 'categorical_c..."
mlp,132415,164000,0.692375,"{'max_steps': 20, 'repeats': 2, 'categorical_c..."
decision_tree,117573,164000,0.549625,"{'max_steps': 20, 'repeats': 2, 'categorical_c..."
logistic_regression,132848,164000,0.695375,"{'max_steps': 20, 'repeats': 2, 'categorical_c..."


,records,members,accuracy,balanced_accuracy,roc_auc,tpr_at_1pct_fpr_roc,threshold,tn,fp,fn,tp,shadow_selected_threshold,tpr_at_shadow_threshold,fpr_at_shadow_threshold
train,11984.0,5958.0,0.614403,0.614635,0.663945,0.049681,0.5004,3457.0,2569.0,2052.0,3906.0,0.624076,0.020309,0.002323
validation,4023.0,2032.0,0.515536,0.515057,0.510761,0.009350,0.5004,932.0,1059.0,890.0,1142.0,0.624076,0.009350,0.009543
test,3993.0,2010.0,0.517155,0.516845,0.520103,0.004478,0.5004,934.0,1049.0,879.0,1131.0,0.624076,0.006468,0.010590


In [46]:
# Query this notebook's original target using its own fitted preprocessor.
target_attack_data, rf_target_walk_diagnostics = collect_target_paper_features(
    model, preprocessor, partitions, config=WALK_CONFIG, seed=42)
display(pd.Series(rf_target_walk_diagnostics, name='target query diagnostics'))
target_attack_metrics, target_membership_scores = evaluate_attack(
    attack_result['classifier'], target_attack_data,
    threshold=attack_result['accuracy_threshold'],
    low_fpr_threshold=attack_result['low_fpr_threshold'])
pd.DataFrame({
    'Adult attack test': attack_result['metrics']['test'],
    'ACS 2018 target MIA': target_attack_metrics,
}).T

Attack feature extraction: 512/20,000
Attack feature extraction: 1,024/20,000
Attack feature extraction: 1,536/20,000
Attack feature extraction: 2,048/20,000
Attack feature extraction: 2,560/20,000
Attack feature extraction: 3,072/20,000
Attack feature extraction: 3,584/20,000
Attack feature extraction: 4,096/20,000
Attack feature extraction: 4,608/20,000
Attack feature extraction: 5,120/20,000
Attack feature extraction: 5,632/20,000
Attack feature extraction: 6,144/20,000
Attack feature extraction: 6,656/20,000
Attack feature extraction: 7,168/20,000
Attack feature extraction: 7,680/20,000
Attack feature extraction: 8,192/20,000
Attack feature extraction: 8,704/20,000
Attack feature extraction: 9,216/20,000
Attack feature extraction: 9,728/20,000
Attack feature extraction: 10,240/20,000
Attack feature extraction: 10,752/20,000
Attack feature extraction: 11,264/20,000
Attack feature extraction: 11,776/20,000
Attack feature extraction: 12,288/20,000
Attack feature extraction: 12,800/20,

record_queries                                                       599762
max_record_queries                                                   820000
walk_censored_fraction                                               0.5652
config                    {'max_steps': 20, 'repeats': 2, 'categorical_c...
Name: target query diagnostics, dtype: object

,records,members,accuracy,balanced_accuracy,roc_auc,tpr_at_1pct_fpr_roc,threshold,tn,fp,fn,tp,shadow_selected_threshold,tpr_at_shadow_threshold,fpr_at_shadow_threshold
Adult attack test,3993.0,2010.0,0.517155,0.516845,0.520103,0.004478,0.5004,934.0,1049.0,879.0,1131.0,0.624076,0.006468,0.01059
ACS 2018 target MIA,20000.0,10000.0,0.507700,0.507700,0.507796,0.009400,0.5004,5277.0,4723.0,5123.0,4877.0,0.624076,0.008400,0.00890
